# Metric Space Visualizer — Plotly Prototyping Notebook

This notebook is a **learning + prototyping sandbox**. Everything you build here gets
copied into `app.py`, the actual Streamlit app you'll deploy and share.

Streamlit apps are `.py` scripts run with `streamlit run app.py` — they don't run
inside a notebook. So think of this notebook as: *figure out the Plotly code here,
where you can see results instantly, then paste the working function into the app.*

### Plotly vs. Matplotlib/Seaborn — quick vocabulary map

| You know (matplotlib/seaborn) | Plotly equivalent |
|---|---|
| `plt.figure()` | `go.Figure()` |
| `plt.plot(x, y)` | `fig.add_trace(go.Scatter(x=x, y=y, mode='lines'))` |
| `plt.scatter(x, y)` | `fig.add_trace(go.Scatter(x=x, y=y, mode='markers'))` |
| `plt.fill_between(...)` | `go.Scatter(..., fill='toself')` |
| `ax.set_title()`, `ax.set_xlim()` | `fig.update_layout(title=..., xaxis=dict(range=...))` |
| `plt.show()` | `fig.show()` |
| `Axes3D` / `ax.plot_surface()` | `go.Surface(x=X, y=Y, z=Z)` |

The big practical difference: Plotly figures are **interactive by default** (zoom, pan,
rotate, hover tooltips) with no extra code — that's the whole reason we're using it
over matplotlib for this project.


In [2]:
import numpy as np
import plotly.graph_objects as go

# If plots don't render inline, uncomment the next two lines:
# import plotly.io as pio
# pio.renderers.default = "notebook"


## 1. Distance functions

Straight from your notes — the general Minkowski / $\ell_p$ distance on $\mathbb{R}^n$:

$$d_p(x, y) = \left( \sum_{k=1}^n |x_k - y_k|^p \right)^{1/p}, \qquad d_\infty(x,y) = \max_k |x_k - y_k|$$

- $p=1$ → Manhattan / L1 (your $d_{3,n}$)
- $p=2$ → Euclidean / L2 (your $d_{1,n}$)
- $p=\infty$ → Chebyshev / max (your $d_{2,n}$)


In [3]:
def minkowski_distance(p1, p2, p):
    """d_p(x, y) for two points in R^n. Use p=np.inf for Chebyshev distance."""
    p1, p2 = np.asarray(p1, dtype=float), np.asarray(p2, dtype=float)
    diff = np.abs(p1 - p2)
    if p == np.inf:
        return diff.max()
    return (diff ** p).sum() ** (1 / p)

# quick sanity check against your notes: two points in R^2
x, y = (0, 0), (3, 4)
print("d1  (Manhattan) :", minkowski_distance(x, y, 1))   # 3 + 4 = 7
print("d2  (Euclidean) :", minkowski_distance(x, y, 2))   # sqrt(9+16) = 5
print("dinf(Chebyshev) :", minkowski_distance(x, y, np.inf))  # max(3,4) = 4


d1  (Manhattan) : 7.0
d2  (Euclidean) : 5.0
dinf(Chebyshev) : 4.0


## 2. Drawing a 2D unit ball boundary

Here's the trick, since a general $p$-norm ball isn't a simple polar circle:

1. Take an angle $\theta$ around a full circle (`np.linspace(0, 2*pi, ...)`), just like you
   would to draw a normal circle with `cos(theta), sin(theta)`.
2. For each direction $(\cos\theta, \sin\theta)$, find the scale factor $s$ so that
   the scaled point sits exactly on the boundary $d_p(s \cdot \text{direction}, 0) = R$.
   Since $d_p$ is homogeneous, $s = R / \big(|\cos\theta|^p + |\sin\theta|^p\big)^{1/p}$.
3. Multiply the direction vector by $s$ — that traces out the boundary of $B(a, R)$.

This is exactly analogous to how you'd plot a circle in matplotlib with
`x = r*np.cos(theta); y = r*np.sin(theta)` — we're just replacing the constant radius
`r` with a direction-dependent radius.


In [4]:
def minkowski_ball_2d(center, R, p, n_points=300):
    """Returns (x, y) arrays tracing the boundary of B(center, R) under d_p in R^2."""
    theta = np.linspace(0, 2 * np.pi, n_points)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    if p == np.inf:
        norm = np.maximum(np.abs(cos_t), np.abs(sin_t))
    else:
        norm = (np.abs(cos_t) ** p + np.abs(sin_t) ** p) ** (1 / p)
    norm = np.where(norm == 0, 1e-12, norm)  # avoid divide-by-zero at odd angles
    scale = R / norm
    x = center[0] + scale * cos_t
    y = center[1] + scale * sin_t
    return x, y

# test: p=2 should give a perfect circle
x, y = minkowski_ball_2d((0, 0), 1, 2)
print(x[:5], y[:5])


[1.         0.99977921 0.99911695 0.99801351 0.99646937] [0.         0.02101245 0.04201562 0.06300024 0.08395704]


In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    fill="toself",              # <- plotly's version of plt.fill_between, but closed-region
    mode="lines",
    line=dict(color="royalblue"),
    fillcolor="rgba(65,105,225,0.3)",
    name="p = 2 (Euclidean)",
))
fig.update_layout(
    title="B((0,0), 1) under d2 — should look like a perfect circle",
    xaxis_title="x1", yaxis_title="x2",
    yaxis=dict(scaleanchor="x", scaleratio=1),  # keeps circles circular (equal aspect ratio)
    width=500, height=500,
)
fig.show()


## 3. Overlaying multiple p-values — the containment picture

This is the single most "impressive" visual for your prof: plotting $B_{d_1}$, $B_{d_2}$,
and $B_{d_\infty}$ at the *same* center and radius on one axes shows the containment
$B_\infty \supseteq B_2 \supseteq B_1$ that underlies norm-equivalence proofs.


In [6]:
fig = go.Figure()
colors = ["crimson", "royalblue", "seagreen", "darkorange", "purple", "black"]
p_values = [1, 1.5, 2, 3, 10, np.inf]
labels = ["p=1 (diamond)", "p=1.5", "p=2 (circle)", "p=3", "p=10", "p=inf (square)"]

for (p, label, color) in zip(p_values, labels, colors):
    x, y = minkowski_ball_2d((0, 0), 1, p)
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(color=color, width=2), name=label))

fig.update_layout(
    title="Unit balls for different p — watch the shape morph diamond -> circle -> square",
    xaxis_title="x1", yaxis_title="x2",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=600, height=600,
)
fig.show()


## 4. Why there's no "slider" cell here

In a plain Plotly notebook you *can* build slider-driven animations using
`fig.update_layout(sliders=[...])` + `frames=[...]`, but that's fiddly and it's
solving a problem Streamlit already solves better: in the actual app, moving a
`st.slider` just **reruns the whole script** with the new value and redraws the figure.
So instead of wrestling with Plotly's native animation API, we keep these functions
plain (just take `p`, `R`, `center` as arguments) and let Streamlit's widgets drive them.
That's why `minkowski_ball_2d` above takes plain arguments rather than building
animation frames — it's designed to be called fresh on every Streamlit rerun.


## 5. Moving to 3D — spherical parametrization

Same idea, one dimension up. Instead of a single angle $\theta$, we need two angles
(polar $\theta \in [0,\pi]$, azimuthal $\phi \in [0, 2\pi]$) to cover a sphere of
directions, exactly like how you'd parametrize a unit sphere for a 3D plot:

$$\hat{d} = (\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta)$$

Then scale each direction vector by $s = R / \|\hat d\|_p$, same as the 2D case.
Plotly's `go.Surface` wants `x, y, z` as 2D grids (from `np.meshgrid`), which is the
direct analogue of matplotlib's `ax.plot_surface(X, Y, Z)`.


In [7]:
def minkowski_ball_3d(center, R, p, n_theta=60, n_phi=60):
    """Returns (x, y, z) 2D grids tracing the surface of B(center, R) under d_p in R^3."""
    theta = np.linspace(0, np.pi, n_theta)
    phi = np.linspace(0, 2 * np.pi, n_phi)
    theta, phi = np.meshgrid(theta, phi)

    dx = np.sin(theta) * np.cos(phi)
    dy = np.sin(theta) * np.sin(phi)
    dz = np.cos(theta)

    if p == np.inf:
        norm = np.maximum(np.maximum(np.abs(dx), np.abs(dy)), np.abs(dz))
    else:
        norm = (np.abs(dx) ** p + np.abs(dy) ** p + np.abs(dz) ** p) ** (1 / p)
    norm = np.where(norm == 0, 1e-12, norm)
    scale = R / norm

    x = center[0] + scale * dx
    y = center[1] + scale * dy
    z = center[2] + scale * dz
    return x, y, z

x3, y3, z3 = minkowski_ball_3d((0, 0, 0), 1, 1)  # p=1 -> should look like an octahedron


In [8]:
fig = go.Figure(data=[go.Surface(x=x3, y=y3, z=z3, colorscale="Blues", opacity=0.9, showscale=False)])
fig.update_layout(
    title="B((0,0,0), 1) under d1 in R^3 — should look like an octahedron",
    scene=dict(xaxis_title="x1", yaxis_title="x2", zaxis_title="x3", aspectmode="cube"),
    width=600, height=600,
)
fig.show()


## 6. Distance calculator sanity check

A tiny feature: given two points, show all three distances side by side. Cheap to add,
reinforces that these are genuinely different numbers for the same two points.


In [9]:
x_pt, y_pt = (0, 0), (3, 4)
for p, label in [(1, "d1"), (2, "d2"), (np.inf, "dinf")]:
    print(f"{label}: {minkowski_distance(x_pt, y_pt, p):.4f}")


d1: 7.0000
d2: 5.0000
dinf: 4.0000


## 7. From here to the Streamlit app

Everything above — `minkowski_distance`, `minkowski_ball_2d`, `minkowski_ball_3d` — is
copied verbatim into `app.py`. The only thing that changes is *where the numbers come
from*: here you typed them directly into function calls; in `app.py`, they come from
`st.slider(...)`, `st.select_slider(...)`, and `st.number_input(...)` in the sidebar.

Run the app with:

```bash
pip install streamlit plotly numpy
streamlit run app.py
```

It'll open in your browser at `localhost:8501`. To share with classmates/your prof
without them installing anything, push the repo to GitHub and deploy for free on
Streamlit Community Cloud (share.streamlit.io) — one click from the repo.
